# Notebook 1: how a decision model walks a knowledge graph

Many questions about a knowledge graph can be answered by following its connections. Take "Who directed the movies that
Tom Hanks acted in?". You start at Tom Hanks, follow *acted in* to his movies, then *directed by* to their directors,
and stop.

At every step there is a list of connections to choose from, and a **decision model** is built for exactly that kind
of choice. You give it a numbered list of options, and it returns its choice and a probability for every option. It
does not write any text.

![Walking the film graph: from Tom Hanks to his films to their directors, with a decision model choosing each step](images/film_walk.svg)

In this notebook you will:

1. send one decision request by hand, and look at the probabilities that come back,
2. send the same request to an LLM, and compare,
3. let a decision model walk a whole question, step by step,
4. read the decision log to see why the walk reached its answer,
5. try your own question.

You need `OPENROUTER_API_KEY` in the `.env` file. Everything in this notebook costs less than one cent.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import utils/ from the repository root

import json
from dataclasses import asdict

import pandas as pd

from utils import DecisionModel, LLM, display, graph, models, walk

driver = graph.connect()
info = graph.ensure_loaded(driver, "metaqa", [])
print(f"The film graph is in Neo4j: {info['entities']:,} entities and {info['facts']:,} facts.")

## The film graph

The graph holds films, the people who acted in, directed and wrote them, and their genres, languages and release years.
Here are Tom Hanks's connections. You can also explore the graph in the Neo4j Browser at http://localhost:7474.

In [ ]:
records, _, _ = driver.execute_query(
    "MATCH (e:KGEntity {key: 'metaqa|kb|Tom Hanks'})-[r:KG_EDGE]-(t) "
    "RETURN r.relation AS relation, CASE WHEN startNode(r) = e THEN 'outgoing' ELSE 'incoming' END AS direction, "
    "count(t) AS connections, collect(t.name)[..4] AS examples ORDER BY connections DESC")
pd.DataFrame([r.data() for r in records])

## Step 1: one decision request, by hand

A request has three parts:

- **the state**: the question, the entities the walk is standing on, and the path so far;
- **the instructions**: the same for every step;
- **the options**: every connection from where the walk stands, numbered. Option 0 is always **STOP**, which means
  "the entities I'm standing on are the answer".

Each option follows one relationship to *all* of its targets. For example, "follow starred_actors (incoming)" moves from
Tom Hanks to every film he acted in.

![What a decision request looks like: the state, the instructions and the numbered options, and the choice and probabilities that come back](images/request_anatomy.svg)

The next cell builds this request from the graph and prints it.

In [ ]:
QUESTION = "Who directed the movies that Tom Hanks acted in?"
START = ["Tom Hanks"]

adriver = graph.connect_async()
question = {"qid": "tom-hanks", "question": QUESTION, "start": START}
rows = await graph.neighbours(adriver, "metaqa", question, START)
options = graph.build_options(rows, START, "metaqa", question)
request = walk.build_request({"question": QUESTION, "current_entities": START, "path_so_far": [], "hop": 0}, options)
print(json.dumps(request, indent=2, ensure_ascii=False))

Now send it to a decision model, d1-ultra from Liquid. It returns a probability for every option.

In [ ]:
decision_model = DecisionModel("liquid/d1-ultra")
async with models.client() as http:
    choice = await decision_model.choose(http, request)

print(f"Chosen option: {choice.option}   cost: ${choice.cost:.6f}   time: {choice.latency_ms} ms")
display.options_table([o["text"] for o in options], asdict(choice))

## Step 2: the same request to an LLM

An LLM writes text, so we ask it for a small **structured answer** instead. It is a Pydantic model with one field,
`option`, which must be one of the offered option numbers. For the four options above, it is the same as writing:

```python
class OptionChoice(BaseModel):
    model_config = ConfigDict(extra="forbid")
    option: Literal["0", "1", "2", "3"]
```

Its JSON schema goes to the LLM as a strict response format, so the LLM can only reply with a valid option. We read
its answer with the same Pydantic model, without parsing any text. An LLM does not return a probability for every
option.

![One request, two kinds of model: the decision model returns a probability for every option, and the LLM returns one option through its Pydantic answer model](images/model_vs_llm.svg)

In [ ]:
from utils import answer_model

Answer = answer_model(tuple(request["questions"]["next_step"]["criteria"]))
print(json.dumps(Answer.model_json_schema(), indent=2))

llm = LLM("openai/gpt-5.6-luna")
async with models.client() as http:
    await llm.check(http)
    llm_choice = await llm.choose(http, request)

print(f"\nThe LLM replied: {llm_choice.reply}")
print(f"That is option {llm_choice.option}: {options[int(llm_choice.option)]['text']}")
print(f"Cost: ${llm_choice.cost:.6f}   (the decision model: ${choice.cost:.6f})")

## Step 3: a whole walk

A walk repeats that step. Get the options, ask the model, follow its choice, and stop when the model chooses STOP. On
this graph a walk may take at most 4 steps.

In [ ]:
result = await walk.walk_question(QUESTION, START, decision_model)

print("Question:", QUESTION)
print("Path:    ", " -> ".join(result["steps"]) or "(stopped at once)")
print(f"Answer:   {len(result['final'])} entities, e.g. {', '.join(result['final'][:6])}")
print(f"Cost:     ${result['cost']:.6f} for {result['calls']} decisions")
display.walk_log(result["decisions"])

## Step 4: read the decision log

The decision log keeps every option the model saw, with its probability. A high probability means the model was sure.
When two options are close, the walk could easily have gone another way, so those steps are the first place to look
when an answer is wrong.

Here are two real steps on this graph: a sure step from the Tom Hanks walk, and a close call from another question.

![Reading the decision log: a sure step, and a close call where the walk took the wrong relationship](images/reading_the_log.svg)

The cell below shows the top four options of each step of your walk.

In [ ]:
pd.concat({f"step {step['hop'] + 1}": display.options_table(step["options"], step["calls"][-1], top=4)
           for step in result["decisions"]})

## Is the answer right?

For this question we can check the walk with a database query, because we know the exact path.

Tom Hanks directed two of his own films. A walk on the film graph never goes back to the entity it started from,
because in MetaQA the question's own entity is never its answer, so the query leaves him out too.

In [ ]:
records, _, _ = driver.execute_query(
    "MATCH (hanks:KGEntity {key: 'metaqa|kb|Tom Hanks'})<-[:KG_EDGE {relation: 'starred_actors'}]-(film)"
    "-[:KG_EDGE {relation: 'directed_by'}]->(director) WHERE director <> hanks "
    "RETURN DISTINCT director.name AS name")
truth = {r["name"] for r in records}
found = set(result["final"])
print(f"Directors of Tom Hanks's films, other than Tom Hanks: {len(truth)}")
print(f"The walk found {len(found & truth)} of them, and {len(found - truth)} other entities.")

## Your turn

Change the question, the starting entity or the model, and run the cell again. The starting entity must be spelled as it
is in the graph. Use `find("...")` to look a name up.

Some ideas:

- "What genres are the films that Christopher Nolan directed?"
- "Who wrote the films that Meryl Streep starred in?"
- "Which languages are the films directed by Akira Kurosawa in?"

You can also try another model, for example `DecisionModel("inception/mercury-decide")` or
`LLM("openai/gpt-5.6-luna")`.

In [ ]:
def find(text: str) -> list[str]:
    records, _, _ = driver.execute_query(
        "MATCH (e:KGEntity {benchmark: 'metaqa'}) WHERE toLower(e.name) CONTAINS toLower($text) "
        "RETURN e.name AS name LIMIT 10", text=text)
    return [r["name"] for r in records]

find("nolan")

In [ ]:
MY_QUESTION = "What genres are the films that Christopher Nolan directed?"
MY_START = ["Christopher Nolan"]
MY_MODEL = DecisionModel("liquid/d1-ultra")

mine = await walk.walk_question(MY_QUESTION, MY_START, MY_MODEL)
print("Path:  ", " -> ".join(mine["steps"]) or "(stopped at once)")
print("Answer:", ", ".join(mine["final"][:15]))
display.walk_log(mine["decisions"])

In [ ]:
await adriver.close()
driver.close()

Next: `2_benchmarking_decision_models.ipynb`, where the models answer hundreds of questions with known answers.